In [29]:
import asyncio
from sqlalchemy.ext.asyncio import AsyncSession, create_async_engine
from sqlalchemy.orm import declarative_base, sessionmaker

from sqlalchemy.future import select
from app.models.log import HttpRequestLog  # import your HttpRequestLog model
from contextlib import asynccontextmanager
from sqlalchemy.orm import Session
from typing import AsyncIterator, List

import os

class ProjectSettings:
    PROJECT_NAME: str = "Anomaly Detection"
    PROJECT_VERSION: str = "1.0.0"

    # Database settings
    POSTGRES_USER: str = os.getenv("POSTGRES_USER", "postgres")
    POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD", "postgres")
    POSTGRES_DB: str = os.getenv("POSTGRES_DB", "anomaly_detection")
    POSTGRES_SERVER: str = os.getenv("POSTGRES_SERVER", "127.0.0.1")
    POSTGRES_PORT: str = os.getenv("POSTGRES_PORT", 5432)  # default postgres port is 5432

    # Connection strings
    DATABASE_URL = f"postgresql+asyncpg://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_SERVER}:{POSTGRES_PORT}/{POSTGRES_DB}"
    SYNC_DB_URL = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_SERVER}:{POSTGRES_PORT}/{POSTGRES_DB}"

projectSettings = ProjectSettings()

engine = create_async_engine(projectSettings.DATABASE_URL, future=True, echo=True, pool_size=20, max_overflow=10)


async_session = sessionmaker(engine, expire_on_commit=False, class_=AsyncSession)

@asynccontextmanager
async def get_db() -> AsyncIterator[AsyncSession]:
    async with async_session() as session:
        yield session


async def fetch_data(db: Session) -> List[HttpRequestLog]:
    result = await db.execute(select(HttpRequestLog))
                              # .offset(skip).limit(limit))
    return result.scalars().all()

# Usage
# data = asyncio.run(fetch_data())
# Instead of asyncio.run(fetch_data()), directly await the coroutine
async with get_db() as db:
    data = await fetch_data(db)


data

2024-01-10 10:27:46,809 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2024-01-10 10:27:46,810 INFO sqlalchemy.engine.Engine [raw sql] ()
2024-01-10 10:27:46,813 INFO sqlalchemy.engine.Engine select current_schema()
2024-01-10 10:27:46,814 INFO sqlalchemy.engine.Engine [raw sql] ()
2024-01-10 10:27:46,816 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2024-01-10 10:27:46,816 INFO sqlalchemy.engine.Engine [raw sql] ()
2024-01-10 10:27:46,817 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2024-01-10 10:27:46,820 INFO sqlalchemy.engine.Engine SELECT logs.id, logs.created_at, logs.updated_at, logs.client_host, logs.request_time, logs.request_method, logs.request_path, logs.http_version, logs.referrer, logs.user_agent, logs.user_id, logs.log_message, logs.cookies, logs.post_params, logs.get_params, logs.body, logs.response_status, logs.process_time 
FROM logs
2024-01-10 10:27:46,820 INFO sqlalchemy.engine.Engine [generated in 0.00070s] ()
2024-01-10 10:27:47,086 I

 ...]

In [3]:
len(data), data[0].__dict__


(8091,
 {'_sa_instance_state': <sqlalchemy.orm.state.InstanceState at 0x7fc22863dc10>,
  'updated_at': None,
  'id': 1,
  'request_time': 1630213188.0,
  'request_method': 'GET',
  'http_version': 'HTTP/1.0',
  'user_agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Safari/537.36',
  'log_message': '-',
  'post_params': '{}',
  'body': None,
  'process_time': 0.0,
  'created_at': datetime.datetime(2024, 1, 9, 18, 20, 3, 229222, tzinfo=datetime.timezone.utc),
  'client_host': '8.20.101.59',
  'request_path': '/',
  'referrer': '-',
  'user_id': None,
  'cookies': '{"fastapi-csrf-token": "4n0RvAnsVOXrWfpBULda9787SOQkU2752j6NAMa0KwFmfE0NMjPtIJcMP6p5ubDg"}',
  'get_params': '{}',
  'response_status': 200})

In [30]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

class DataLoader:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(max_features=100)
        self.scaler = MinMaxScaler()

    def fit_transform(self, logs):
        # Concatenate textual fields for TF-IDF
        text_data = [" ".join([log.request_method, log.user_agent, log.request_path, log.cookies or '', log.referrer or '', log.log_message or '', log.post_params or '', log.get_params or '',]) for log in logs]

        # Vectorize textual data
        text_features = self.vectorizer.fit_transform(text_data).toarray()

        # Extract and scale numerical fields
        numerical_data = np.array([[log.response_status, log.process_time, log.user_id or 0] for log in logs])
        numerical_features = self.scaler.fit_transform(numerical_data)

        # Combine text and numerical features
        features = np.hstack((text_features, numerical_features))
        return features

    def transform(self, logs):
        text_data = [" ".join([log.request_method, log.user_agent, log.request_path, log.cookies or '', log.referrer or '', log.log_message or '', log.post_params or '', log.get_params or '',]) for log in logs]
        text_features = self.vectorizer.transform(text_data).toarray()

        numerical_data = np.array([[log.response_status, log.process_time, log.user_id or 0] for log in logs])
        numerical_features = self.scaler.transform(numerical_data)

        features = np.hstack((text_features, numerical_features))
        return features

  # 'logs' is your list of HttpRequestLog instances


In [31]:
# Usage
data_loader = DataLoader()
X_train = data_loader.fit_transform(data)

In [ ]:
# # For new log entries
# X_new = data_loader.transform(new_logs)
# # Now, X_new is ready to be fed into your trained model for anomaly detection.


In [32]:
import torch
from torch import nn

class Autoencoder(nn.Module):
    def __init__(self, input_size):
        super(Autoencoder, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(True),
            nn.Linear(128, 64),
            nn.ReLU(True),
            nn.Linear(64, 32),
            nn.ReLU(True),
            nn.Linear(32, 16)  # Compressed representation
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(True),
            nn.Linear(32, 64),
            nn.ReLU(True),
            nn.Linear(64, 128),
            nn.ReLU(True),
            nn.Linear(128, input_size),
            nn.Tanh()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x


In [33]:
from torch.utils.data import DataLoader, TensorDataset

# Convert the numpy array to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)

# Create dataset and dataloader for batch processing
train_dataset = TensorDataset(X_train_tensor)
train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)


In [34]:
# Model Initialization
input_size = X_train.shape[1]  # Number of features in your data
model = Autoencoder(input_size)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Training Loop
num_epochs = 50
for epoch in range(num_epochs):
    for data in train_loader:
        inputs = data[0]
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, inputs)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')


Epoch [1/50], Loss: 0.0085
Epoch [2/50], Loss: 0.0072
Epoch [3/50], Loss: 0.0064
Epoch [4/50], Loss: 0.0049
Epoch [5/50], Loss: 0.0036
Epoch [6/50], Loss: 0.0041
Epoch [7/50], Loss: 0.0032
Epoch [8/50], Loss: 0.0031
Epoch [9/50], Loss: 0.0019
Epoch [10/50], Loss: 0.0020
Epoch [11/50], Loss: 0.0018
Epoch [12/50], Loss: 0.0016
Epoch [13/50], Loss: 0.0018
Epoch [14/50], Loss: 0.0012
Epoch [15/50], Loss: 0.0013
Epoch [16/50], Loss: 0.0012
Epoch [17/50], Loss: 0.0015
Epoch [18/50], Loss: 0.0013
Epoch [19/50], Loss: 0.0011
Epoch [20/50], Loss: 0.0016
Epoch [21/50], Loss: 0.0014
Epoch [22/50], Loss: 0.0009
Epoch [23/50], Loss: 0.0010
Epoch [24/50], Loss: 0.0007
Epoch [25/50], Loss: 0.0008
Epoch [26/50], Loss: 0.0008
Epoch [27/50], Loss: 0.0006
Epoch [28/50], Loss: 0.0006
Epoch [29/50], Loss: 0.0006
Epoch [30/50], Loss: 0.0008
Epoch [31/50], Loss: 0.0006
Epoch [32/50], Loss: 0.0003
Epoch [33/50], Loss: 0.0006
Epoch [34/50], Loss: 0.0006
Epoch [35/50], Loss: 0.0008
Epoch [36/50], Loss: 0.0007
E

In [35]:
import re
from app.models.log import HttpRequestLog  # Import your HttpRequestLog model

def get_log_data(log):
    re_exp = '(^[0-9]+\.[0-9]+\.[0-9]+\.[0-9]+) ([0-9]*) \"(.*?) (.*?)\" (.*?) (.*?) \"(.*?)\" \"(.*?)\" \"(.*?)\" \"(.*?)\" (.*?) ({.*?}) ({.*?}) ({.*?})'
    print(log,"log")
    match = re.search(re_exp, log)
    if not match:
        print("NO MATCH FOUNND")
        return None

    return {
        'client_host': match.group(1),
        'request_time': float(match.group(2)) if match.group(2) else None,
        'request_method': match.group(3),
        'request_path': match.group(4),
        'http_version': match.group(5),
        'response_status': int(match.group(6).split()[0]) if match.group(6).split() else None,
        'process_time': float(match.group(6).split()[1]) if len(match.group(6).split()) > 1 else None,
        'referrer': match.group(7),
        'user_agent': match.group(8),
        'user_id': int(match.group(10)) if match.group(10).isdigit() else None,
        'log_message': match.group(11),
        'cookies': match.group(12),
        'post_params': match.group(13),
        'get_params': match.group(14),
        'body': None
    }

def create_http_request_log_instance_from_string(log_string):
    log_data = get_log_data(log_string)
    if log_data is None:
        return None  # Or handle the error as per your application's need
    print(log_data,111)
    # Create an instance of HttpRequestLog with the extracted data
    log_instance = HttpRequestLog(
        client_host=log_data['client_host'],
        request_time=log_data['request_time'],
        request_method=log_data['request_method'],
        request_path=log_data['request_path'],
        http_version=log_data['http_version'],
        response_status=log_data['response_status'],
        process_time=log_data['process_time'],
        referrer=log_data['referrer'],
        user_agent=log_data['user_agent'],
        user_id=log_data['user_id'],
        log_message=log_data['log_message'],
        cookies=log_data['cookies'],
        post_params=log_data['post_params'],
        get_params=log_data['get_params'],
        body=log_data['body']
    )

    return log_instance


In [36]:
example_log = """8.20.101.59 1630213368 "POST /dashboard/users/create" HTTP/1.0 302 32 "http://localhost:3000/dashboard/users/create" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Safari/537.36" "-" "6" - {"fastapi-csrf-token": "xwHvkU1SzNqx6S22sPHgJ1qFy3ST7GBSgBdOa7ALsyy5ycdIFcmjTbfvRhRYedFP", "sessionId": "9nrhkxte0z38n8k7bfjkyi2v1zgbxu3h"} {"csrfmiddlewaretoken": "3Q4OMEsz7icX5dIsQ5AxBgYYVaO6lB2rMVA7CR1s03kvxxT83sfALqNOeoNbs86o", "username": "aashutosh", "email": "aashutosh9881@gmail.com", "hospital": "5"} {} """
# Assuming you have a function to create a HttpRequestLog instance from a log string
example_log_instance = create_http_request_log_instance_from_string(example_log)


example_log_instance

8.20.101.59 1630213368 "POST /dashboard/users/create" HTTP/1.0 302 32 "http://localhost:3000/dashboard/users/create" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Safari/537.36" "-" "6" - {"fastapi-csrf-token": "xwHvkU1SzNqx6S22sPHgJ1qFy3ST7GBSgBdOa7ALsyy5ycdIFcmjTbfvRhRYedFP", "sessionId": "9nrhkxte0z38n8k7bfjkyi2v1zgbxu3h"} {"csrfmiddlewaretoken": "3Q4OMEsz7icX5dIsQ5AxBgYYVaO6lB2rMVA7CR1s03kvxxT83sfALqNOeoNbs86o", "username": "aashutosh", "email": "aashutosh9881@gmail.com", "hospital": "5"} {}  log
{'client_host': '8.20.101.59', 'request_time': 1630213368.0, 'request_method': 'POST', 'request_path': '/dashboard/users/create', 'http_version': 'HTTP/1.0', 'response_status': 302, 'process_time': 32.0, 'referrer': 'http://localhost:3000/dashboard/users/create', 'user_agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Safari/537.36', 'user_id': 6, 'log_message':

In [37]:
def classify_anomaly(log):
    example_log_instance = create_http_request_log_instance_from_string(log)
    # Preprocess the example log data
    X_example = data_loader.transform([example_log_instance])  # Transform expects a list of instances
    X_example_tensor = torch.FloatTensor(X_example)
    with torch.no_grad():
        reconstructed = model(X_example_tensor)
        loss = nn.functional.mse_loss(reconstructed, X_example_tensor, reduction='none')
        reconstruction_error = loss.mean().item()
        print('error: ',reconstruction_error)
    anomaly_threshold = 0.5  # Set this based on your model's performance and dataset
    is_anomalous = reconstruction_error > anomaly_threshold
    print(f"Reconstruction Error: {reconstruction_error}")
    print(f"Anomalous: {is_anomalous}")
    return is_anomalous


In [38]:
classify_anomaly(example_log)

8.20.101.59 1630213368 "POST /dashboard/users/create" HTTP/1.0 302 32 "http://localhost:3000/dashboard/users/create" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Safari/537.36" "-" "6" - {"fastapi-csrf-token": "xwHvkU1SzNqx6S22sPHgJ1qFy3ST7GBSgBdOa7ALsyy5ycdIFcmjTbfvRhRYedFP", "sessionId": "9nrhkxte0z38n8k7bfjkyi2v1zgbxu3h"} {"csrfmiddlewaretoken": "3Q4OMEsz7icX5dIsQ5AxBgYYVaO6lB2rMVA7CR1s03kvxxT83sfALqNOeoNbs86o", "username": "aashutosh", "email": "aashutosh9881@gmail.com", "hospital": "5"} {}  log
{'client_host': '8.20.101.59', 'request_time': 1630213368.0, 'request_method': 'POST', 'request_path': '/dashboard/users/create', 'http_version': 'HTTP/1.0', 'response_status': 302, 'process_time': 32.0, 'referrer': 'http://localhost:3000/dashboard/users/create', 'user_agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Safari/537.36', 'user_id': 6, 'log_message':

False

In [39]:
aalog = '''8.20.101.59 1630213360 "POST /dashboard/accounts/login/?next=%2Fdashboard%2Fhospitals%2F" HTTP/1.0 403 1203 "http://localhost:3000/dashboard/accounts/login/?next=/dashboard/hospitals/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.13; rv:91.0) Gecko/20100101 Firefox/91.0" "-" "-" - {"fastapi-csrf-token": "f1JgAcJ5KsyiXcgRV9a0aac2MOuVtLJHgxUpZWpUcwnQemAmSuiRAWD9kyT8T5n8"} {"csrfmiddlewaretoken": "eLhVPDxeibfpqwzeIgZdQqGyOxbMEwVHfhs4end3Kf4XHGTJFB74gc7FmhAZ4Qz8'))) AND 9147=9147 AND ((('FKBE' LIKE 'FKBE", "username": "sdf", "password": "sdf"} {"next": "/dashboard/hospitals/"} 
'''
classify_anomaly(aalog)

8.20.101.59 1630213360 "POST /dashboard/accounts/login/?next=%2Fdashboard%2Fhospitals%2F" HTTP/1.0 403 1203 "http://localhost:3000/dashboard/accounts/login/?next=/dashboard/hospitals/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.13; rv:91.0) Gecko/20100101 Firefox/91.0" "-" "-" - {"fastapi-csrf-token": "f1JgAcJ5KsyiXcgRV9a0aac2MOuVtLJHgxUpZWpUcwnQemAmSuiRAWD9kyT8T5n8"} {"csrfmiddlewaretoken": "eLhVPDxeibfpqwzeIgZdQqGyOxbMEwVHfhs4end3Kf4XHGTJFB74gc7FmhAZ4Qz8'))) AND 9147=9147 AND ((('FKBE' LIKE 'FKBE", "username": "sdf", "password": "sdf"} {"next": "/dashboard/hospitals/"} 
 log
{'client_host': '8.20.101.59', 'request_time': 1630213360.0, 'request_method': 'POST', 'request_path': '/dashboard/accounts/login/?next=%2Fdashboard%2Fhospitals%2F', 'http_version': 'HTTP/1.0', 'response_status': 403, 'process_time': 1203.0, 'referrer': 'http://localhost:3000/dashboard/accounts/login/?next=/dashboard/hospitals/', 'user_agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.13; rv:91.0) Gecko/20100101

False

In [40]:
classify_anomaly('''110.44.127.184 1630370634 "GET /?search=hello%22%29%29%20AND%205826%3DCONVERT%28INT%2C%28SELECT%20CHAR%28113%29%2BCHAR%28118%29%2BCHAR%28106%29%2BCHAR%2898%29%2BCHAR%28113%29%2B%28SELECT%20%28CASE%20WHEN%20%285826%3D5826%29%20THEN%20CHAR%2849%29%20ELSE%20CHAR%2848%29%20END%29%29%2BCHAR%28113%29%2BCHAR%28122%29%2BCHAR%28118%29%2BCHAR%28120%29%2BCHAR%28113%29%29%29%20AND%20%28%28%22WueG%22%3D%22WueG&district=47" HTTP/1.0 200 1849526 "http://logcovid.prixa.live:80/" "sqlmap/1.5.9#stable (http://sqlmap.org)" "-" "-" - {"csrftoken": "0x07FM9YdaX47ezkcAJGTo6On1CXu3IZ8tnT4LFaw7TifVUr9Kp6DHrD5dcdm0lb"} {} {"search": "hello\")) AND 5826=CONVERT(INT,(SELECT CHAR(113)+CHAR(118)+CHAR(106)+CHAR(98)+CHAR(113)+(SELECT (CASE WHEN (5826=5826) THEN CHAR(49) ELSE CHAR(48) END))+CHAR(113)+CHAR(122)+CHAR(118)+CHAR(120)+CHAR(113))) AND ((\"WueG\"=\"WueG", "district": "47"} ''')

110.44.127.184 1630370634 "GET /?search=hello%22%29%29%20AND%205826%3DCONVERT%28INT%2C%28SELECT%20CHAR%28113%29%2BCHAR%28118%29%2BCHAR%28106%29%2BCHAR%2898%29%2BCHAR%28113%29%2B%28SELECT%20%28CASE%20WHEN%20%285826%3D5826%29%20THEN%20CHAR%2849%29%20ELSE%20CHAR%2848%29%20END%29%29%2BCHAR%28113%29%2BCHAR%28122%29%2BCHAR%28118%29%2BCHAR%28120%29%2BCHAR%28113%29%29%29%20AND%20%28%28%22WueG%22%3D%22WueG&district=47" HTTP/1.0 200 1849526 "http://logcovid.prixa.live:80/" "sqlmap/1.5.9#stable (http://sqlmap.org)" "-" "-" - {"csrftoken": "0x07FM9YdaX47ezkcAJGTo6On1CXu3IZ8tnT4LFaw7TifVUr9Kp6DHrD5dcdm0lb"} {} {"search": "hello")) AND 5826=CONVERT(INT,(SELECT CHAR(113)+CHAR(118)+CHAR(106)+CHAR(98)+CHAR(113)+(SELECT (CASE WHEN (5826=5826) THEN CHAR(49) ELSE CHAR(48) END))+CHAR(113)+CHAR(122)+CHAR(118)+CHAR(120)+CHAR(113))) AND (("WueG"="WueG", "district": "47"}  log
{'client_host': '110.44.127.184', 'request_time': 1630370634.0, 'request_method': 'GET', 'request_path': '/?search=hello%22%29%29%20AND

False

In [34]:
with torch.no_grad():
    reconstructed = model(X_example_tensor)
    loss = nn.functional.mse_loss(reconstructed, X_example_tensor, reduction='none')
    reconstruction_error = loss.mean().item()


In [35]:
anomaly_threshold = 0.1  # Set this based on your model's performance and dataset
is_anomalous = reconstruction_error > anomaly_threshold
print(f"Reconstruction Error: {reconstruction_error}")
print(f"Anomalous: {is_anomalous}")


Reconstruction Error: 0.00954094436019659
Anomalous: False
